<td>   <a target="_blank" href="https://labelbox.com" ><img src="https://labelbox.com/blog/content/images/2021/02/logo-v4.svg" width=256/></a></td>


<td>
<a href="https://colab.research.google.com/github/Labelbox/labelbox-python/blob/develop/examples/annotation_import/audio_temporal.ipynb" target="_blank"><img
src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>
</td>

<td>
<a href="https://github.com/Labelbox/labelbox-python/tree/develop/examples/annotation_import/audio_temporal.ipynb" target="_blank"><img
src="https://img.shields.io/badge/GitHub-100000?logo=github&logoColor=white" alt="GitHub"></a>
</td>


# Audio Temporal Annotation Import

This notebook demonstrates how to create and upload **temporal audio annotations** - annotations that are tied to specific time ranges in audio files.

## What are Temporal Audio Annotations?

Temporal audio annotations allow you to:
- **Transcribe speech** with precise timestamps ("Hello world" from 2.5s to 4.1s)
- **Identify speakers** in specific segments ("John speaking" from 10s to 15s)
- **Detect sound events** with time ranges ("Dog barking" from 30s to 32s)
- **Classify audio quality** for segments ("Clear audio" from 0s to 10s)

## Supported Temporal Annotations

- **AudioClassificationAnnotation**: Radio, checklist, and text classifications for time ranges
- **AudioObjectAnnotation**: Text entities (transcriptions) for time ranges

## Key Features

- **Time-based API**: Use seconds for user-friendly input
- **Frame-based storage**: Internally uses milliseconds (1 frame = 1ms)
- **MAL compatible**: Works with existing Model-Assisted Labeling pipeline
- **UI compatible**: Uses existing video timeline components

## Import Methods

- **Model-Assisted Labeling (MAL)**: Upload pre-annotations for labeler review
- **Label Import**: Upload ground truth labels directly


## Setup


In [ ]:
%pip install -q "labelbox[data]"


In [ ]:
import labelbox as lb
import labelbox.types as lb_types
import uuid
from typing import List


### Replace with your API key
Guides on [Create an API key](https://docs.labelbox.com/docs/create-an-api-key)


In [ ]:
# Add your api key
API_KEY = ""
client = lb.Client(api_key=API_KEY)


## Creating Temporal Audio Annotations

### Audio Classification Annotations

Use `AudioClassificationAnnotation` for classifications tied to specific time ranges. The interface now accepts milliseconds directly for precise timing control.


In [ ]:
# Speaker identification for a time range
speaker_annotation = lb_types.AudioClassificationAnnotation.from_time_range(
    start_ms=2500,  # Start at 2500 milliseconds (2.5 seconds)
    end_ms=4100,    # End at 4100 milliseconds (4.1 seconds)
    name="speaker_id",
    value=lb_types.Radio(answer=lb_types.ClassificationAnswer(name="john"))
)

print(f"Speaker annotation frame: {speaker_annotation.frame}ms")
print(f"Speaker annotation start time: {speaker_annotation.start_time}s")


In [ ]:
# Audio quality assessment for a segment
quality_annotation = lb_types.AudioClassificationAnnotation.from_time_range(
    start_ms=0,
    end_ms=10000,
    name="audio_quality",
    value=lb_types.Checklist(answer=[
        lb_types.ClassificationAnswer(name="clear_audio"),
        lb_types.ClassificationAnswer(name="no_background_noise")
    ])
)

# Emotion detection for a segment
emotion_annotation = lb_types.AudioClassificationAnnotation.from_time_range(
    start_ms=5200,
    end_ms=8700,
    name="emotion",
    value=lb_types.Radio(answer=lb_types.ClassificationAnswer(name="happy"))
)


### Audio Object Annotations

Use `AudioObjectAnnotation` for text entities like transcriptions tied to specific time ranges. The interface now accepts milliseconds directly for precise timing control.


In [ ]:
# Transcription with precise timestamps
transcription_annotation = lb_types.AudioObjectAnnotation.from_time_range(
    start_ms=2500,
    end_ms=4100,
    name="transcription",
    value=lb_types.TextEntity(text="Hello, how are you doing today?")
)

print(f"Transcription frame: {transcription_annotation.frame}ms")
print(f"Transcription text: {transcription_annotation.value.text}")


In [ ]:
# Sound event detection
sound_event_annotation = lb_types.AudioObjectAnnotation.from_time_range(
    start_ms=10000,
    end_ms=12500,
    name="sound_event",
    value=lb_types.TextEntity(text="Dog barking in background")
)

# Multiple transcription segments
transcription_segments = [
    lb_types.AudioObjectAnnotation.from_time_range(
        start_ms=0, end_ms=2300,
        name="transcription",
        value=lb_types.TextEntity(text="Welcome to our podcast.")
    ),
    lb_types.AudioObjectAnnotation.from_time_range(
        start_ms=2500, end_ms=5800,
        name="transcription", 
        value=lb_types.TextEntity(text="Today we're discussing AI advancements.")
    ),
    lb_types.AudioObjectAnnotation.from_time_range(
        start_ms=6000, end_ms=9200,
        name="transcription",
        value=lb_types.TextEntity(text="Let's start with machine learning basics.")
    )
]


## Use Cases and Examples

### Use Case 1: Podcast Transcription with Speaker Identification


In [ ]:
# Complete podcast annotation with speakers and transcriptions
podcast_annotations = [
    # Host introduction
    lb_types.AudioClassificationAnnotation.from_time_range(
        start_ms=0, end_ms=5000,
        name="speaker_id",
        value=lb_types.Radio(answer=lb_types.ClassificationAnswer(name="host"))
    ),
    lb_types.AudioObjectAnnotation.from_time_range(
        start_ms=0, end_ms=5000,
        name="transcription",
        value=lb_types.TextEntity(text="Welcome to Tech Talk, I'm your host Sarah.")
    ),
    
    # Guest response
    lb_types.AudioClassificationAnnotation.from_time_range(
        start_ms=5200, end_ms=8500,
        name="speaker_id",
        value=lb_types.Radio(answer=lb_types.ClassificationAnswer(name="guest"))
    ),
    lb_types.AudioObjectAnnotation.from_time_range(
        start_ms=5200, end_ms=8500,
        name="transcription",
        value=lb_types.TextEntity(text="Thanks for having me, Sarah!")
    ),
    
    # Audio quality assessment
    lb_types.AudioClassificationAnnotation.from_time_range(
        start_ms=0, end_ms=10000,
        name="audio_quality",
        value=lb_types.Radio(answer=lb_types.ClassificationAnswer(name="excellent"))
    )
]

print(f"Created {len(podcast_annotations)} podcast annotations")


### Use Case 2: Call Center Quality Analysis


In [ ]:
# Call center analysis with sentiment and quality metrics
call_center_annotations = [
    # Customer sentiment analysis
    lb_types.AudioClassificationAnnotation.from_time_range(
        start_ms=0, end_ms=30000,
        name="customer_sentiment",
        value=lb_types.Radio(answer=lb_types.ClassificationAnswer(name="frustrated"))
    ),
    
    # Agent performance
    lb_types.AudioClassificationAnnotation.from_time_range(
        start_ms=30000, end_ms=60000,
        name="agent_performance",
        value=lb_types.Checklist(answer=[
            lb_types.ClassificationAnswer(name="professional_tone"),
            lb_types.ClassificationAnswer(name="resolved_issue"),
            lb_types.ClassificationAnswer(name="followed_script")
        ])
    ),
    
    # Key phrases extraction
    lb_types.AudioObjectAnnotation.from_time_range(
        start_ms=15000, end_ms=18000,
        name="key_phrase",
        value=lb_types.TextEntity(text="I want to speak to your manager")
    ),
    
    lb_types.AudioObjectAnnotation.from_time_range(
        start_ms=45000, end_ms=48000,
        name="key_phrase",
        value=lb_types.TextEntity(text="Thank you for your patience")
    )
]

print(f"Created {len(call_center_annotations)} call center annotations")


### Use Case 3: Music and Sound Event Detection


In [ ]:
# Music analysis and sound event detection
music_annotations = [
    # Musical instruments
    lb_types.AudioClassificationAnnotation.from_time_range(
        start_ms=0, end_ms=30000,
        name="instruments",
        value=lb_types.Checklist(answer=[
            lb_types.ClassificationAnswer(name="piano"),
            lb_types.ClassificationAnswer(name="violin"),
            lb_types.ClassificationAnswer(name="drums")
        ])
    ),
    
    # Genre classification
    lb_types.AudioClassificationAnnotation.from_time_range(
        start_ms=0, end_ms=60000,
        name="genre",
        value=lb_types.Radio(answer=lb_types.ClassificationAnswer(name="classical"))
    ),
    
    # Sound events
    lb_types.AudioObjectAnnotation.from_time_range(
        start_ms=25000, end_ms=27000,
        name="sound_event",
        value=lb_types.TextEntity(text="Applause from audience")
    ),
    
    lb_types.AudioObjectAnnotation.from_time_range(
        start_ms=45000, end_ms=46500,
        name="sound_event",
        value=lb_types.TextEntity(text="Door closing in background")
    )
]

print(f"Created {len(music_annotations)} music annotations")


## Uploading Audio Temporal Prelabels

### Step 1: Import Audio Data into Catalog


In [ ]:
# Create dataset with audio file
global_key = "sample-audio-temporal-" + str(uuid.uuid4())

asset = {
    "row_data": "https://storage.googleapis.com/labelbox-datasets/audio-sample-data/sample-audio-1.mp3",
    "global_key": global_key,
}

dataset = client.create_dataset(name="audio_temporal_demo_dataset")
task = dataset.create_data_rows([asset])
task.wait_till_done()
print("Errors:", task.errors)
print("Failed data rows:", task.failed_data_rows)


### Step 2: Create Ontology with Temporal Audio Tools

Your ontology must include the tools and classifications that match your annotation names.


In [ ]:
ontology_builder = lb.OntologyBuilder(
    tools=[
        # Text entity tools for transcriptions and sound events
        lb.Tool(tool=lb.Tool.Type.TEXT_ENTITY, name="transcription"),
        lb.Tool(tool=lb.Tool.Type.TEXT_ENTITY, name="sound_event"),
        lb.Tool(tool=lb.Tool.Type.TEXT_ENTITY, name="key_phrase"),
    ],
    classifications=[
        # Speaker identification
        lb.Classification(
            class_type=lb.Classification.Type.RADIO,
            name="speaker_id",
            scope=lb.Classification.Scope.INDEX,  # Frame-based classification
            options=[
                lb.Option(value="host"),
                lb.Option(value="guest"),
                lb.Option(value="john"),
                lb.Option(value="sarah"),
            ],
        ),
        
        # Audio quality assessment
        lb.Classification(
            class_type=lb.Classification.Type.CHECKLIST,
            name="audio_quality",
            scope=lb.Classification.Scope.INDEX,
            options=[
                lb.Option(value="clear_audio"),
                lb.Option(value="no_background_noise"),
                lb.Option(value="good_volume"),
                lb.Option(value="excellent"),
            ],
        ),
        
        # Emotion detection
        lb.Classification(
            class_type=lb.Classification.Type.RADIO,
            name="emotion",
            scope=lb.Classification.Scope.INDEX,
            options=[
                lb.Option(value="happy"),
                lb.Option(value="sad"),
                lb.Option(value="angry"),
                lb.Option(value="neutral"),
            ],
        ),
        
        # Customer sentiment (for call center example)
        lb.Classification(
            class_type=lb.Classification.Type.RADIO,
            name="customer_sentiment",
            scope=lb.Classification.Scope.INDEX,
            options=[
                lb.Option(value="satisfied"),
                lb.Option(value="frustrated"),
                lb.Option(value="angry"),
                lb.Option(value="neutral"),
            ],
        ),
        
        # Agent performance (for call center example)
        lb.Classification(
            class_type=lb.Classification.Type.CHECKLIST,
            name="agent_performance",
            scope=lb.Classification.Scope.INDEX,
            options=[
                lb.Option(value="professional_tone"),
                lb.Option(value="resolved_issue"),
                lb.Option(value="followed_script"),
                lb.Option(value="empathetic_response"),
            ],
        ),
        
        # Music instruments (for music example)
        lb.Classification(
            class_type=lb.Classification.Type.CHECKLIST,
            name="instruments",
            scope=lb.Classification.Scope.INDEX,
            options=[
                lb.Option(value="piano"),
                lb.Option(value="violin"),
                lb.Option(value="drums"),
                lb.Option(value="guitar"),
            ],
        ),
        
        # Music genre
        lb.Classification(
            class_type=lb.Classification.Type.RADIO,
            name="genre",
            scope=lb.Classification.Scope.INDEX,
            options=[
                lb.Option(value="classical"),
                lb.Option(value="jazz"),
                lb.Option(value="rock"),
                lb.Option(value="pop"),
            ],
        ),
    ],
)

ontology = client.create_ontology(
    "Audio Temporal Annotations Ontology",
    ontology_builder.asdict(),
    media_type=lb.MediaType.Audio,
)

print(f"Created ontology: {ontology.name}")


### Step 3: Create Project and Setup Editor


In [ ]:
# Create project
project = client.create_project(
    name="Audio Temporal Annotations Demo",
    media_type=lb.MediaType.Audio
)

# Connect ontology to project
project.setup_editor(ontology)

print(f"Created project: {project.name}")


### Step 4: Create Batch and Add Data


In [ ]:
# Create batch
batch = project.create_batch(
    "audio-temporal-batch-" + str(uuid.uuid4())[:8],
    global_keys=[global_key],
    priority=5,
)

print(f"Created batch: {batch.name}")


### Step 5: Upload Temporal Audio Annotations via MAL

Now we'll upload our temporal audio annotations using the Model-Assisted Labeling pipeline.


In [ ]:
# Create label with temporal audio annotations
# Using the podcast example annotations
label = lb_types.Label(
    data={"global_key": global_key},
    annotations=podcast_annotations
)

print(f"Created label with {len(podcast_annotations)} temporal annotations")
print("Annotation types:")
for i, annotation in enumerate(podcast_annotations):
    ann_type = type(annotation).__name__
    if hasattr(annotation, 'frame'):
        time_info = f"at {annotation.start_time}s (frame {annotation.frame})"
    else:
        time_info = "global"
    print(f"  {i+1}. {ann_type} '{annotation.name}' {time_info}")


In [ ]:
# Upload via MAL (Model-Assisted Labeling)
upload_job = lb.MALPredictionImport.create_from_objects(
    client=client,
    project_id=project.uid,
    name=f"audio_temporal_mal_{str(uuid.uuid4())[:8]}",
    predictions=[label],
)

upload_job.wait_until_done()
print("Upload completed!")
print("Errors:", upload_job.errors)
print("Status:", upload_job.statuses)


## NDJSON Format Examples

Temporal audio annotations serialize to NDJSON format similar to video annotations, with frame-based timing.


In [ ]:
# Let's examine how temporal audio annotations serialize to NDJSON
from labelbox.data.serialization.ndjson.label import NDLabel
import json

# Serialize our label to NDJSON format
ndjson_generator = NDLabel.from_common([label])
ndjson_objects = list(ndjson_generator)

print(f"Generated {len(ndjson_objects)} NDJSON objects")
print("\nNDJSON Examples:")
print("=" * 50)

for i, obj in enumerate(ndjson_objects[:3]):  # Show first 3 examples
    print(f"\nObject {i+1}:")
    # Convert to dict for pretty printing
    obj_dict = obj.dict(exclude_none=True)
    print(json.dumps(obj_dict, indent=2))


### Comparison with Video Annotations

Audio temporal annotations use the same frame-based structure as video annotations:


In [ ]:
print("Frame-based Structure Comparison:")
print("=" * 40)

# Audio: 1 frame = 1 millisecond
audio_annotation = lb_types.AudioClassificationAnnotation.from_time_range(
    start_ms=2500, end_ms=4100,
    name="test", value=lb_types.Text(answer="test")
)

print(f"Audio Annotation:")
print(f"  Time: 2500ms → Frame: {audio_annotation.frame} (milliseconds)")
print(f"  Frame rate: 1000 frames/second (1 frame = 1ms)")

print(f"\nVideo Annotation (for comparison):")
print(f"  Time: 2.5s → Frame: depends on video frame rate")
print(f"  Frame rate: varies (e.g., 30 fps = 30 frames/second)")

print(f"\nBoth use the same NDJSON structure with 'frame' field")


## Best Practices

### 1. Time Precision
- Audio temporal annotations use millisecond precision (1 frame = 1ms)
- Use the `from_time_range()` method with millisecond-based input for precise timing control
- Frame values are set directly: `frame = start_ms`

### 2. Ontology Alignment
- Ensure annotation `name` fields match your ontology tool/classification names
- Use `scope=lb.Classification.Scope.INDEX` for frame-based classifications
- Text entity tools work for transcriptions and sound event descriptions

### 3. Segment Organization
- Use `segment_index` to group related annotations
- Segments help organize timeline view in the UI
- Each segment can contain multiple annotation types

### 4. Performance Optimization
- Batch multiple labels in a single MAL import for better performance
- Use appropriate time ranges - avoid overly granular segments
- Consider audio file length when planning annotation density


## Cleanup (Optional)


In [ ]:
# Uncomment to clean up resources
# project.delete()
# dataset.delete()
# ontology.delete()


## Summary

This notebook demonstrated:

1. **Creating temporal audio annotations** using `AudioClassificationAnnotation` and `AudioObjectAnnotation`
2. **Millisecond-based API** with `from_time_range()` for precise timing control
3. **Multiple use cases**: podcasts, call centers, music analysis
4. **MAL import pipeline** for uploading temporal prelabels
5. **NDJSON serialization** compatible with existing video infrastructure
6. **Best practices** for ontology setup and performance optimization

### Key Benefits:
- **No UI changes needed** - uses existing video timeline components
- **Frame-based precision** - 1ms accuracy for audio timing
- **Seamless integration** - works with existing MAL and Label Import pipelines
- **Flexible annotation types** - supports classifications and text entities with timestamps
- **Direct millisecond input** - precise timing control without conversion overhead

### Next Steps:
1. Upload your temporal audio annotations using this notebook as a template
2. Review annotations in the Labelbox editor (uses video timeline UI)
3. Export annotated data for model training or analysis
4. Integrate with your audio processing pipeline
